In [1]:
import numpy as np
from scipy.signal import resample_poly, spectrogram
import matplotlib.pyplot as plt
import sys
import os
import torch
# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa, MultiBAMv3

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)


c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [ ]:
sf = 9
bw = 125000
fs = 1000000
lora_init = LoRa(sf, bw)

## HOW TO LOAD WEIGHT
layers = [256*15, 1024, 256] # <-- must match training

multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)
# for i, bam in enumerate(multi_bam.bams):
#     bam.W = np.load(f"weight_3840_1024_256_model2/weights_layer_{i}.npy")

for i, bam in enumerate(multi_bam.bams):
    w_np = np.load(f"weight/weights_layer_{i}.npy")
    bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)
    print(f"Loaded layer {i}: shape {bam.W.shape}")


In [ ]:
symbol1 = 0
symbol2 = 200
# EXAMPLE OF COMPRESS and DECOMPRESS

x1 = lora_init.gen_symbol_fs(symbol1, sf=sf, bw=bw, Fs=int(bw*8))  # you had Fs=int(bw*8)=1e6
snr = -15
x = lora_init.awgn_iq(x1,snr)
x_down,fs_new_down = downsampling(x,fs,4)
image_ori_noise,null,null = create_spectrogram_npy(x_down,fs_new_down,0,0,1,None)

x2 = apply_cfo(x, Fs=fs_new_down, freq_offset_hz=300)
x2 = apply_phase_noise(x2, Fs=fs_new_down, linewidth_hz=50)
x2 = multipath_rayleigh(x2, [0, 5], 6)
x2 = band_limited_noise(x2, Fs=fs_new_down, low_hz=110e3, high_hz=120e3, snr_db=20)
x2 = quantize_iq(x2, nbits=8)
x2 = time_varying_rayleigh(x2,10,fs_new_down)
x2 =  hard_clip(x2, max_amp=0.1)

#### STEP 1 PREPROCESS : DOWNSAMPLING ########
x_ds,fs_new = downsampling(x2,fs,4)

#### STEP 2 Create Spectrogram ########
aa,bb,cc = create_spectrogram_npy(x_ds,fs_new,0,0,1,None)

#### STEP 3 FLATTEN INPUT ########
flat = aa.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
out1 = multi_bam.compress(flat)
print(out1.shape)
out2 = multi_bam.decompress(out1)
print(out2.shape)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flat = out2.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec = reco_flat.reshape(256, 15)
print(reco_spec.shape)
array_a = [image_ori_noise,aa,reco_spec]

show_multiple_spectrograms(array_a,["ori noise","broken","decompress"],3)

In [ ]:
from scipy.signal import stft
# x = signal lora
# nperseg = 128
# noverlap = 64
# nfft = 512
snr = -20
nperseg = 128
noverlap = 64
nfft = 512
window = "hann"
f11, t11, Zxx11 = stft(
    x_ds, fs=fs_new, window=window, nperseg=nperseg,
    noverlap=noverlap, nfft=nfft, padded=False, boundary=None,
    return_onesided=False
)
f = np.fft.fftshift(f11)
Sxx = np.fft.fftshift(Zxx11,axes=0)
# Sxx_dB = 10 * np.log10(np.real(Sxx) + 1e-12)
# Sxx_dB_i = 10 * np.log10(np.imag(Sxx) + 1e-12)
mask = (f >= -bw/2) & (f < bw/2)
Sxx_crop = Sxx[mask, :]

print(np.min(np.real(Sxx_crop)))
print(np.max(np.real(Sxx_crop)))
print(np.min(np.imag(Sxx_crop)))
print(np.max(np.imag(Sxx_crop)))
print(Sxx_crop.shape)
tes_dummy = lora_init.gen_symbol_fs(symbol2,sf,bw,False,fs)
tes_dummy_low = lora_init.awgn_iq(tes_dummy,snr)
tes_dummy_2,f_d_new = downsampling(tes_dummy_low,fs,4)
q,w,e = create_spectrogram_npy(tes_dummy_2,f_d_new,1,1,1,None)
print(q.shape)
print("Sea")
# Normalize to 0–1 range
sxx_real = np.real(Sxx)
sxx_imag = np.imag(Sxx)
Sxx_norm = (sxx_real - sxx_real.min()) / (sxx_real.max() - sxx_real.min())
Sxx_norm_i = (sxx_imag - sxx_imag.min()) / (sxx_imag.max() - sxx_imag.min())
# # ---- CROP to ±BW/2 ----
mask = (f >= -bw/2) & (f < bw/2)
Sxx_crop = Sxx_norm[mask, :]

the_signal =  q
#### STEP 3 FLATTEN INPUT ########
flat = the_signal.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
out1 = multi_bam.compress(flat)
print(out1.shape)
out2 = multi_bam.decompress(out1)
print(out2.shape)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flat = out2.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec = reco_flat.reshape(256, 15)
print(reco_spec.shape)
# Normalize to 0–1 range
# # ---- CROP to ±BW/2 ----
mask_i = (f >= -bw/2) & (f < bw/2)
Sxx_crop_i = Sxx_norm_i[mask_i, :]
flat_i = Sxx_crop_i.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
out1 = multi_bam.compress(flat_i)
print(out1.shape)
out2 = multi_bam.decompress(out1)
print(out2.shape)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flat = out2.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec_i = reco_flat.reshape(256, 15)
array_b = [q,reco_spec]
check_simmilarity(reco_spec,q)
show_multiple_spectrograms(array_b,["original","decompress"],2)
# Sxx = np.fft.fftshift(Zxx11, axes=0)
# print(Zxx11)
# print(np.abs(Zxx11))
# print(np.angle(Zxx11))

# print(Sx.shape)
# print(Sx)
# sx = np.fft.fftshift(Sx,axes = 0)
# from scipy.signal import istft
# show_spectrogram_from_npy(np.real(Zxx11))
# show_spectrogram_from_npy(np.imag(Zxx11))
# show_spectrogram_from_npy(np.abs(Zxx11))

# show_spectrogram_from_npy(np.imag(Zxx11))
# # Reconstruct
# _, x_rec = istft(
#     Zxx11, fs=fs, window=window, nperseg=nperseg,
#     noverlap=noverlap, nfft=nfft, input_onesided=False
# )
# print(x_rec.shape)
# # check error
# err = np.max(np.abs(x_ds - x_rec))
# print("Max absolute error:", err)
# print("Relative RMSE:", np.linalg.norm(x_ds - x_rec) / np.linalg.norm(x_ds))


# PLOT_SPECGRAM(x_ds,128,"a",32,fs/4)
# PLOT_SPECGRAM(x_rec,128,"a",32,fs/4)
